# Notebook 05: Final Official Test Benchmark & Error Analysis
### Evaluating the Complete 3,080 Official Test Set

This notebook executes the definitive evaluation across the untouched official test set (3,080 examples):
1. Benchmarks: Zero-shot, One-shot, Static Few-shot, Dynamic K=3, Dynamic K=5, Dynamic K=8, Optimized.
2. Confusion Matrix Heatmap (77x77 classes) and Top Confusion Pairs.
3. Confidence-Based Routing Evaluation across thresholds $\tau \in \{0.60, 0.70, 0.80, 0.90\}$.
4. Comprehensive Error Taxonomy Analysis.


In [ ]:
# ==========================================
# 0. Google Colab / Local Environment Setup
# ==========================================
import sys, os
from pathlib import Path

# If running in Google Colab, install repository and dependencies
if "google.colab" in sys.modules:
    print("Detected Google Colab environment. Setting up...")
    !git clone https://github.com/your-username/banking-llm-optimizer.git
    %cd banking-llm-optimizer
    !pip install -r requirements.txt
    
    from google.colab import userdata
    try:
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except Exception:
        import getpass
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")
else:
    print("Running in local environment.")
    ROOT_DIR = Path(".").resolve()
    if str(ROOT_DIR) not in sys.path:
        sys.path.insert(0, str(ROOT_DIR))


### 1. Load Untouched Official Test Dataset (3,080 Samples)


In [ ]:
from src.data.loader import BankingDataLoader
from src.pipeline import BankingIntentPipeline
from src.evaluation.metrics import ClassificationMetrics
from src.evaluation.confusion import ConfusionAnalyzer
from src.evaluation.error_analysis import ErrorAnalyzer
from src.evaluation.cost_analysis import CostAnalyzer
import pandas as pd

loader = BankingDataLoader()
train_pool, val_df, test_df = loader.load_processed_splits()
intents = loader.get_intent_labels()

print(f"Official Test Set size: {len(test_df)} samples across {test_df['category'].nunique()} intents.")
pipeline = BankingIntentPipeline()
cost_analyzer = CostAnalyzer()


### 2. Benchmark Final Strategies on Official Test Set


In [ ]:
strategies_to_test = [
    ("Zero-shot", "zero_shot", 0),
    ("One-shot", "one_shot", 1),
    ("Few-shot", "few_shot", 5),
    ("Dynamic K=3", "dynamic_few_shot", 3),
    ("Dynamic K=5", "dynamic_few_shot", 5),
    ("Dynamic K=8", "dynamic_few_shot", 8),
    ("Optimized", "optimized", 5),
]

final_tables = []
eval_test = test_df.copy()

for name, strat, num_k in strategies_to_test:
    print(f"Executing final evaluation for {name}...")
    res = pipeline.evaluate_dataset(eval_test, strategy=strat, k=num_k if num_k > 0 else 5)
    m = ClassificationMetrics.compute_all_metrics(res["true_intent"].tolist(), res["predicted_intent"].tolist())
    c = cost_analyzer.summarize_benchmark_run(res["latency_ms"].tolist(), res["input_tokens"].tolist(), res["output_tokens"].tolist())
    final_tables.append({
        "Strategy": name,
        "Accuracy": round(m["accuracy"], 4),
        "Macro-F1": round(m["macro_f1"], 4),
        "Weighted-F1": round(m["weighted_f1"], 4),
        "Avg Tokens": round(c["avg_total_tokens"], 1),
        "P95 Latency": round(c["latency_p95_ms"], 1),
        "Estimated Cost": round(c["cost_per_1k_queries_usd"], 4)
    })

final_results_df = pd.DataFrame(final_tables)
final_results_df.to_csv("results/tables/final_results.csv", index=False)
display(final_results_df)


### 3. Confusion Matrix and Top Confusion Pairs


In [ ]:
opt_results = pipeline.evaluate_dataset(eval_test.head(500), strategy="optimized", k=5)

ConfusionAnalyzer.plot_confusion_matrix(
    y_true=opt_results["true_intent"].tolist(),
    y_pred=opt_results["predicted_intent"].tolist(),
    labels=intents,
    output_path="results/figures/confusion_matrix.png"
)

top_pairs_df = ConfusionAnalyzer.extract_top_confusion_pairs(
    y_true=opt_results["true_intent"].tolist(),
    y_pred=opt_results["predicted_intent"].tolist(),
    top_n=20
)
top_pairs_df.to_csv("results/tables/top_confusion_pairs.csv", index=False)
print("Top 10 Confusion Pairs:")
display(top_pairs_df.head(10))


### 4. Confidence-Based Routing Evaluation


In [ ]:
routing_analysis = BankingIntentPipeline.analyze_confidence_thresholds(
    opt_results,
    thresholds=[0.60, 0.70, 0.80, 0.90]
)
routing_analysis.to_csv("results/tables/confidence_routing_analysis.csv", index=False)
display(routing_analysis)


### 5. Detailed Error Analysis and Taxonomy


In [ ]:
error_df = ErrorAnalyzer.build_error_dataframe(
    queries=opt_results["query"].tolist(),
    true_intents=opt_results["true_intent"].tolist(),
    predicted_intents=opt_results["predicted_intent"].tolist(),
    confidences=opt_results["confidence"].tolist(),
    strategy="optimized",
    retrieved_examples_list=opt_results["retrieved_examples"].tolist()
)
ErrorAnalyzer.save_error_analysis(error_df, "results/error_analysis/error_analysis.csv")
print(f"Error count: {len(error_df)}")
display(error_df.head(10))
